In [ ]:
%%bash
pip install -q --no-cache-dir "transformers==4.44.2" "accelerate==0.34.2" sentencepiece datasets evaluate

In [ ]:
%%bash
pip install -q --upgrade transformers tokenizers accelerate datasets evaluate sentencepiece

In [ ]:
!pip install -q sacrebleu sacremoses

# Config


In [ ]:
CONFIG = {
    "model_name": "UBC-NLP/AraT5v2-base-1024",
    "train_file": "/kaggle/input/datasets/ahmedalaeb/samer-processed-data/samer_train.parquet",
    "dev_file": "/kaggle/input/datasets/ahmedalaeb/samer-processed-data/samer_dev.parquet",
    "output_dir": "/kaggle/working/arat5_checkpoints",
    "export_path": "/kaggle/working/final_arat5_simplified",
    "predictions_file": "/kaggle/working/results/samer_model_dev_predictions.csv",
    "seed": 42,
    "eval_subset_size": 500,
    "early_stopping_patience": 3,
}

# Tokenizer

In [ ]:
import os
import shutil
import json
import torch
from transformers import EarlyStoppingCallback
from huggingface_hub import hf_hub_download
from transformers import (
    T5Tokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from datasets import load_dataset

clean_tok_dir = "/kaggle/working/clean_tokenizer"
os.makedirs(clean_tok_dir, exist_ok=True)
model_name = CONFIG["model_name"]

sp_file = hf_hub_download(repo_id=model_name, filename="spiece.model")
shutil.copy(sp_file, os.path.join(clean_tok_dir, "spiece.model"))

config_payload = {
    "model_max_length": 1024,
    "tokenizer_class": "T5Tokenizer",
    "legacy": True
}
with open(os.path.join(clean_tok_dir, "tokenizer_config.json"), "w", encoding="utf-8") as f:
    json.dump(config_payload, f)

tokenizer = T5Tokenizer.from_pretrained(clean_tok_dir, local_files_only=True)
print(f"Tokenizer initialized successfully | Vocab size: {len(tokenizer)}")

# Model

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.config.tie_word_embeddings = False

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
)

print(f"Model ready on: {device} | Dimensions: {model.shared.weight.shape}")

# Data

In [ ]:
data_files = {
    "train": "/kaggle/input/datasets/ahmedalaeb/samer-processed-data/samer_train.parquet",
    "validation": "/kaggle/input/datasets/ahmedalaeb/samer-processed-data/samer_dev.parquet"
}

raw_datasets = load_dataset("parquet", data_files=data_files)

def preprocess_function(examples):
    inputs = ["بسّط: " + str(text) for text in examples["original_text"]]
    targets = [str(text) for text in examples["simplified_text"]]
    
    model_inputs = tokenizer(inputs, max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=256, truncation=True, padding="max_length")
    
    labels["input_ids"] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

cols_to_remove = raw_datasets["train"].column_names

tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=cols_to_remove
)

print(f"Data ready | Train: {len(tokenized_datasets['train'])} | Validation: {len(tokenized_datasets['validation'])}")

SEED = 42
EVAL_SUBSET_SIZE = 500

import random
random.seed(SEED)

full_dev_size = len(tokenized_datasets["validation"])
eval_subset_size = min(EVAL_SUBSET_SIZE, full_dev_size)

subset_indices = random.sample(range(full_dev_size), eval_subset_size)

dev_subset_for_training = tokenized_datasets["validation"].select(subset_indices)
dev_sources_subset = raw_datasets["validation"].select(subset_indices)["original_text"]
dev_sources_subset = [str(t) for t in dev_sources_subset]

print(f"Eval subset for training: {len(dev_subset_for_training)} rows "
      f"(of {full_dev_size} total dev rows)")

# Evaluation Metrics: SARI, BLEU & Copy Rate

In [ ]:
import evaluate

sari_metric = evaluate.load("sari")
bleu_metric = evaluate.load("bleu")

def compute_copy_rate(sources, predictions):
    matches = sum(1 for s, p in zip(sources, predictions) if s.strip() == p.strip())
    return (matches / len(sources)) * 100.0 if len(sources) > 0 else 0.0

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    predictions = np.where(predictions >= 0, predictions, tokenizer.pad_token_id)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    sari_result = sari_metric.compute(
        sources=dev_sources_subset,
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels],
    )
    bleu_result = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels],
    )
    copy_rate = compute_copy_rate(dev_sources_subset, decoded_preds)

    return {
        "sari": sari_result["sari"],
        "bleu": bleu_result["bleu"],
        "copy_rate": copy_rate
    }

# Traning

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback


training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG["output_dir"],
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    learning_rate=1e-4,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_steps=100,
    predict_with_generate=True,
    generation_max_length=CONFIG.get("max_length", 256),
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=3,   
    report_to="none",
    metric_for_best_model="sari",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=CONFIG["seed"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=dev_subset_for_training,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["early_stopping_patience"])],
)


trainer.train()


print("\nRunning single-pass prediction & evaluation on FULL dev set...")

trainer.compute_metrics = None

predict_output = trainer.predict(tokenized_datasets["validation"])


preds = predict_output.predictions
if isinstance(preds, tuple):
    preds = preds[0]
preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
labels = np.where(predict_output.label_ids != -100, predict_output.label_ids, tokenizer.pad_token_id)

decoded_preds_final = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
decoded_labels_final = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
full_dev_sources = [str(t) for t in raw_datasets["validation"]["original_text"]]


final_sari = sari_metric.compute(
    sources=full_dev_sources,
    predictions=decoded_preds_final,
    references=[[l] for l in decoded_labels_final]
)["sari"]

final_bleu = bleu_metric.compute(
    predictions=decoded_preds_final,
    references=[[l] for l in decoded_labels_final]
)["bleu"]

final_copy_rate = compute_copy_rate(full_dev_sources, decoded_preds_final)

print(f"Final Full Dev Metrics -> SARI: {final_sari:.2f} | BLEU: {final_bleu:.2f} | Copy Rate: {final_copy_rate:.2f}%")


dev_ids = (
    raw_datasets["validation"]["ID"]
    if "ID" in raw_datasets["validation"].column_names
    else list(range(len(full_dev_sources)))
)

predictions_df = pd.DataFrame({
    "ID": dev_ids,
    "original_text": full_dev_sources,
    "reference": decoded_labels_final,
    "prediction": decoded_preds_final,
})

os.makedirs(os.path.dirname(CONFIG["predictions_file"]), exist_ok=True)
predictions_df.to_csv(CONFIG["predictions_file"], index=False, encoding="utf-8-sig")
print(f"Saved {len(predictions_df)} predictions to {CONFIG['predictions_file']}")


os.makedirs("results", exist_ok=True)
with open("results/samer_model_dev.md", "w", encoding="utf-8") as f:
    f.write("# SAMER Model - Dev Evaluation Results\n\n")
    f.write(f"- **Final SARI (Full Dev)**: {final_sari:.2f}\n")
    f.write(f"- **BLEU**: {final_bleu:.2f}\n")
    f.write(f"- **Copy Rate**: {final_copy_rate:.2f}%\n")
print("Results markdown written to results/samer_model_dev.md")


export_path = CONFIG["export_path"]
trainer.save_model(export_path)
tokenizer.save_pretrained(export_path)
print(f"Final model saved successfully to: {export_path}")

# Uploading Model to Hugging Face

In [ ]:
from transformers import AutoModelForSeq2SeqLM
from huggingface_hub import login

login(token="HF_TOKEN")

export_path = CONFIG["export_path"]
repo_id = "samer-arat5v2-base-simplification"

final_model = AutoModelForSeq2SeqLM.from_pretrained(export_path)
final_model.push_to_hub(repo_id)

print("The model (with full weights) has been successfully uploaded to Hugging Face.")